# NeurIPS Conference - Clinical Trials Matching Pipeline

This notebook creates a data pipeline to:
1. Load NeurIPS conference papers (2008-2022)
2. Match clinical trial sponsor firm names in paper titles/abstracts
3. Upload results to the MISR database

**Output Tables:**
- `NeurIPS_Papers`: All NeurIPS papers from 2008-2022
- `NeurIPS_Clinical_Trials`: Papers where clinical trial firms were mentioned

## 1. Setup and Imports

In [1]:
import pandas as pd
import numpy as np
import json
import re
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# Database utilities
from db_utils import MySQLDatabase

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 100)

print("Imports complete.")

Imports complete.


## 2. Load and Process NeurIPS Data

In [2]:
# Load NeurIPS JSON data
with open('data/neurips.json/neurips.json', 'r') as f:
    neurips_raw = json.load(f)

print(f"Total papers in JSON: {len(neurips_raw)}")

# Helper to safely convert year to int
def get_year(paper):
    y = paper.get('year', 0)
    if isinstance(y, str):
        try:
            return int(y)
        except:
            return 0
    return y

# Filter for 2008-2022
papers_filtered = [p for p in neurips_raw if 2008 <= get_year(p) <= 2022]
print(f"Papers from 2008-2022: {len(papers_filtered)}")

Total papers in JSON: 16746
Papers from 2008-2022: 13405


In [3]:
# Transform to DataFrame with required schema
def process_paper(paper, idx):
    """Process a single paper into the required format."""
    authors = paper.get('authors', [])
    abstract = paper.get('abstract', '')
    
    # Handle "Abstract Unavailable" as NULL
    if abstract == 'Abstract Unavailable' or not abstract:
        abstract = None
    
    return {
        'paper_id': idx + 1,  # 1-indexed
        'conference_name': 'NeurIPS',
        'year': get_year(paper),
        'paper_title': paper.get('title', ''),
        'abstract': abstract,
        'authors': json.dumps(authors),  # Store as JSON string
        'author_order': json.dumps(list(range(1, len(authors) + 1))),  # [1, 2, 3, ...]
        'author_affiliations': None,  # Not available in source data
        'paper_url': paper.get('url', ''),
        'keywords': None,  # Not available in source data
        'acknowledgements': None,  # Not available in source data
        'full_text': None  # Not available in source data
    }

# Process all papers
processed_papers = [process_paper(p, i) for i, p in enumerate(papers_filtered)]
df_neurips = pd.DataFrame(processed_papers)

print(f"\nDataFrame shape: {df_neurips.shape}")
print(f"\nColumns: {list(df_neurips.columns)}")
df_neurips.head()


DataFrame shape: (13405, 12)

Columns: ['paper_id', 'conference_name', 'year', 'paper_title', 'abstract', 'authors', 'author_order', 'author_affiliations', 'paper_url', 'keywords', 'acknowledgements', 'full_text']


,paper_id,conference_name,year,paper_title,abstract,authors,author_order,author_affiliations,paper_url,keywords,acknowledgements,full_text
0,1,NeurIPS,2008,Near-minimax recursive density estimation on the binary hypercube,This paper describes a recursive estimation procedure for multivariate binary densities using or...,"[""Maxim Raginsky"", ""Svetlana Lazebnik"", ""Rebecca Willett"", ""Jorge Silva""]","[1, 2, 3, 4]",None,https://papers.nips.cc/paper/2008/hash/00411460f7c92d2124a67ea0f4cb5f85-Abstract.html,None,None,None
1,2,NeurIPS,2008,Translated Learning: Transfer Learning across Different Feature Spaces,None,"[""Wenyuan Dai"", ""Yuqiang Chen"", ""Gui-rong Xue"", ""Qiang Yang"", ""Yong Yu""]","[1, 2, 3, 4, 5]",None,https://papers.nips.cc/paper/2008/hash/0060ef47b12160b9198302ebdb144dcf-Abstract.html,None,None,None
2,3,NeurIPS,2008,Learning Bounded Treewidth Bayesian Networks,"With the increased availability of data for complex domains, it is desirable to learn Bayesian n...","[""Gal Elidan"", ""Stephen Gould""]","[1, 2]",None,https://papers.nips.cc/paper/2008/hash/006f52e9102a8d3be2fe5614f42ba989-Abstract.html,None,None,None
3,4,NeurIPS,2008,Local Gaussian Process Regression for Real Time Online Model Learning,"Learning in real-time applications, e.g., online approximation of the inverse dynamics model for...","[""Duy Nguyen-tuong"", ""Jan Peters"", ""Matthias Seeger""]","[1, 2, 3]",None,https://papers.nips.cc/paper/2008/hash/01161aaa0b6d1345dd8fe4e481144d84-Abstract.html,None,None,None
4,5,NeurIPS,2008,Grouping Contours Via a Related Image,Contours have been established in the biological and computer vision literatures as a compact ye...,"[""Praveen Srinivasan"", ""Liming Wang"", ""Jianbo Shi""]","[1, 2, 3]",None,https://papers.nips.cc/paper/2008/hash/013a006f03dbc5392effeb8f18fda755-Abstract.html,None,None,None


## 3. Data Quality Summary (Prior to Analysis)

In [4]:
print("=" * 60)
print("DATA QUALITY REPORT: NeurIPS Papers (2008-2022)")
print("=" * 60)
print(f"\nTotal papers: {len(df_neurips):,}")
print(f"Year range: {df_neurips['year'].min()} to {df_neurips['year'].max()}")

print("\n" + "-" * 60)
print("FIELD AVAILABILITY (NULL/Missing Statistics):")
print("-" * 60)

# Calculate missing percentages
total = len(df_neurips)
fields = ['year', 'paper_title', 'abstract', 'authors', 'author_order', 
          'author_affiliations', 'paper_url', 'keywords', 'acknowledgements', 'full_text']

missing_report = []
for field in fields:
    null_count = df_neurips[field].isnull().sum()
    null_pct = (null_count / total) * 100
    available_pct = 100 - null_pct
    missing_report.append({
        'Field': field,
        'Available': f"{available_pct:.1f}%",
        'Missing (NULL)': f"{null_pct:.1f}%",
        'NULL Count': null_count
    })
    
df_missing = pd.DataFrame(missing_report)
print(df_missing.to_string(index=False))

print("\n" + "-" * 60)
print("PAPERS BY YEAR:")
print("-" * 60)
year_counts = df_neurips['year'].value_counts().sort_index()
for year, count in year_counts.items():
    print(f"  {year}: {count:,} papers")

print("\n" + "=" * 60)

DATA QUALITY REPORT: NeurIPS Papers (2008-2022)

Total papers: 13,405
Year range: 2008 to 2022

------------------------------------------------------------
FIELD AVAILABILITY (NULL/Missing Statistics):
------------------------------------------------------------
              Field Available Missing (NULL)  NULL Count
               year    100.0%           0.0%           0
        paper_title    100.0%           0.0%           0
           abstract     99.4%           0.6%          84
            authors    100.0%           0.0%           0
       author_order    100.0%           0.0%           0
author_affiliations      0.0%         100.0%       13405
          paper_url    100.0%           0.0%           0
           keywords      0.0%         100.0%       13405
   acknowledgements      0.0%         100.0%       13405
          full_text      0.0%         100.0%       13405

------------------------------------------------------------
PAPERS BY YEAR:
-------------------------------

## 4. Load Clinical Trial Sponsors

In [5]:
# Load clinical trials data
df_trials = pd.read_csv('final_sample_clinical_trials_information.csv')

print(f"Clinical trials loaded: {len(df_trials):,}")
print(f"Unique sponsors: {df_trials['sponsor_name'].nunique()}")
print(f"Year range: {df_trials['start_year'].min()} to {df_trials['start_year'].max()}")

# Get unique sponsor names
unique_sponsors = df_trials['sponsor_name'].unique().tolist()
print(f"\nFirst 10 sponsors: {unique_sponsors[:10]}")

Clinical trials loaded: 9,428
Unique sponsors: 691
Year range: 2008 to 2021

First 10 sponsors: ['UCB Pharma', 'Pfizer', 'Eli Lilly and Company', 'Amgen', 'Mersana Therapeutics', 'Organon and Co', 'UCB Pharma SA', 'GlaxoSmithKline', 'Bayer', 'Allergan']


## 5. Create Canonical Name Dictionary

Build a mapping from each sponsor to normalized search terms, handling:
- Legal suffixes (Inc., Ltd., Corp., etc.)
- Common abbreviations
- Subsidiary relationships

In [6]:
# Legal suffixes to remove for normalization
LEGAL_SUFFIXES = [
    r'\bInc\.?\b', r'\bIncorporated\b', r'\bLtd\.?\b', r'\bLimited\b',
    r'\bCorp\.?\b', r'\bCorporation\b', r'\bLLC\b', r'\bL\.L\.C\.\b',
    r'\bCo\.?\b', r'\bCompany\b', r'\bS\.?A\.?\b', r'\bA/S\b',
    r'\bAG\b', r'\bGmbH\b', r'\bplc\b', r'\bPLC\b', r'\bN\.?V\.?\b',
    r'\bB\.?V\.?\b', r'\bKG\b', r'\bS\.?r\.?l\.?\b', r'\bS\.?p\.?A\.?\b'
]

# Common words that should NOT trigger matches (too generic)
COMMON_WORDS = {
    'applied', 'human', 'global', 'center', 'precision', 'advanced', 'research',
    'science', 'sciences', 'therapeutics', 'therapy', 'life', 'health', 'medical',
    'pharmaceutical', 'biosciences', 'bio', 'international', 'technologies',
    'technology', 'systems', 'imaging', 'diagnostics', 'innovations', 'solutions',
    'network', 'digital', 'clinical', 'molecular', 'genomics', 'genetics',
    'analytics', 'computing', 'software', 'data', 'learning', 'neural', 'deep',
    'machine', 'artificial', 'intelligence', 'vision', 'natural', 'language',
    'achieve', 'target', 'edge', 'threshold', 'core', 'spark', 'fusion', 'signal',
    'synergy', 'catalyst', 'discovery', 'insight', 'forward', 'horizon',
    'evolution', 'revolution', 'dynamics', 'spectrum', 'atlas', 'compass', 'bridge',
    'group', 'united', 'general', 'first', 'new', 'national', 'american', 'world',
    'pharma', 'pharm', 'drug', 'drugs', 'medicine', 'medicines', 'healthcare',
    'biopharmaceutical', 'biopharmaceuticals', 'biotech', 'biotechnology'
}

# Known major pharma companies with their aliases (these are safe to match)
KNOWN_PHARMA = {
    'pfizer': ['pfizer'],
    'glaxosmithkline': ['glaxosmithkline', 'gsk'],
    'bristol-myers squibb': ['bristol-myers squibb', 'bristol myers squibb'],
    'johnson & johnson': ['johnson johnson'],  # Note: "janssen" could be a person name
    'astrazeneca': ['astrazeneca'],
    'novartis': ['novartis'],
    'merck': ['merck'],  # Be careful - could match person names
    'sanofi': ['sanofi'],
    'hoffmann-la roche': ['hoffmann-la roche', 'roche'],
    'eli lilly': ['eli lilly', 'lilly'],
    'abbvie': ['abbvie'],
    'amgen': ['amgen'],
    'gilead sciences': ['gilead'],
    'biogen': ['biogen'],
    'regeneron': ['regeneron'],
    'bayer': ['bayer'],
    'takeda': ['takeda'],
    'boehringer ingelheim': ['boehringer ingelheim', 'boehringer'],
    'novo nordisk': ['novo nordisk'],
    'allergan': ['allergan'],
    'celgene': ['celgene'],
    'genentech': ['genentech'],
    'vertex pharmaceuticals': ['vertex pharmaceuticals'],  # "vertex" alone is too common
    'alexion': ['alexion'],
    'illumina': ['illumina'],
    # Tech companies that might have clinical trials
    'google': ['google', 'deepmind'],
    'microsoft': ['microsoft'],
    'ibm': ['ibm watson'],
    'nvidia': ['nvidia'],
}

def normalize_text(text):
    """Normalize text for matching: lowercase, remove punctuation, extra spaces."""
    if not text:
        return ''
    text = text.lower()
    # Remove legal suffixes
    for suffix in LEGAL_SUFFIXES:
        text = re.sub(suffix, '', text, flags=re.IGNORECASE)
    # Remove geographic info in parentheses
    text = re.sub(r'\([^)]*\)', '', text)
    # Remove punctuation except hyphens
    text = re.sub(r'[^\w\s-]', ' ', text)
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def is_generic_name(name):
    """Check if a name is too generic (composed only of common words)."""
    words = name.lower().split()
    non_common = [w for w in words if w not in COMMON_WORDS and len(w) >= 4]
    return len(non_common) == 0

def build_search_variants(sponsor_name):
    """Build list of search variants for a sponsor name."""
    variants = set()
    normalized = normalize_text(sponsor_name)
    
    # First check if this matches a known pharma company
    for known_name, aliases in KNOWN_PHARMA.items():
        if known_name in normalized or any(alias in normalized for alias in aliases):
            # Use the known aliases for matching
            for alias in aliases:
                if len(alias) >= 6:  # Only add sufficiently long aliases
                    variants.add(alias)
            return list(variants)
    
    # For unknown companies, only use full normalized name if it's specific enough
    if normalized and len(normalized) >= 10 and not is_generic_name(normalized):
        variants.add(normalized)
    
    # Handle subsidiary patterns
    if 'subsidiary of' in sponsor_name.lower():
        parts = re.split(r',?\s*(?:a\s+)?(?:wholly[- ]owned\s+)?subsidiary of\s*', 
                        sponsor_name, flags=re.IGNORECASE)
        for part in parts:
            norm_part = normalize_text(part)
            if norm_part and len(norm_part) >= 10 and not is_generic_name(norm_part):
                variants.add(norm_part)
    
    return list(variants)

# Build canonical name dictionary
canonical_firms = {}
for sponsor in unique_sponsors:
    variants = build_search_variants(sponsor)
    if variants:
        canonical_firms[sponsor] = variants

print(f"Built canonical names for {len(canonical_firms)} sponsors")
print(f"(Filtered out {len(unique_sponsors) - len(canonical_firms)} sponsors with generic names)")

print("\nExample mappings for major pharma:")
sample_sponsors = ['Pfizer', 'GlaxoSmithKline', 'Bristol-Myers Squibb', 
                   'Eli Lilly and Company', 'Hoffmann-La Roche', 'AstraZeneca']
for sponsor in sample_sponsors:
    if sponsor in canonical_firms:
        print(f"  {sponsor}: {canonical_firms[sponsor]}")

Built canonical names for 507 sponsors
(Filtered out 184 sponsors with generic names)

Example mappings for major pharma:
  Pfizer: ['pfizer']
  GlaxoSmithKline: ['glaxosmithkline']
  Bristol-Myers Squibb: ['bristol-myers squibb', 'bristol myers squibb']
  Eli Lilly and Company: ['eli lilly']
  Hoffmann-La Roche: ['hoffmann-la roche']
  AstraZeneca: ['astrazeneca']


## 6. Firm Matching Function

In [7]:
def find_firm_matches(title, abstract, canonical_firms, min_match_length=6):
    """
    Search title and abstract for firm name mentions.
    
    Args:
        title: Paper title
        abstract: Paper abstract
        canonical_firms: Dict mapping sponsor_name -> list of search variants
        min_match_length: Minimum characters for a valid match (avoid false positives)
    
    Returns:
        List of dicts with match info: 
        [{'firm': sponsor_name, 'location': 'title'|'abstract'|'both', 'matched_text': str}]
    """
    matches = []
    
    title_norm = normalize_text(title or '')
    abstract_norm = normalize_text(abstract or '')
    
    for sponsor, variants in canonical_firms.items():
        in_title = False
        in_abstract = False
        matched_variant = None
        
        for variant in variants:
            if len(variant) < min_match_length:
                continue
            
            # Always use word boundary matching to avoid partial matches
            pattern = r'\b' + re.escape(variant) + r'\b'
            
            if re.search(pattern, title_norm):
                in_title = True
                matched_variant = variant
            if re.search(pattern, abstract_norm):
                in_abstract = True
                matched_variant = variant
            
            if in_title or in_abstract:
                break
        
        if in_title or in_abstract:
            location = 'both' if (in_title and in_abstract) else ('title' if in_title else 'abstract')
            matches.append({
                'firm': sponsor,
                'location': location,
                'matched_text': matched_variant
            })
    
    return matches

# Test the function with sample papers
test_cases = [
    ("Deep Learning for Drug Discovery at Pfizer", 
     "We present methods developed at Pfizer Research for molecular optimization."),
    ("Neural Networks for Genomics",
     "This work was done in collaboration with Novartis and GlaxoSmithKline."),
    ("A New Architecture for Vision",
     "No pharmaceutical companies mentioned here."),
    ("DeepMind's AlphaFold Predicts Protein Structures",
     "Google DeepMind researchers developed a breakthrough method.")
]

print("Test matches:")
print("-" * 60)
for title, abstract in test_cases:
    matches = find_firm_matches(title, abstract, canonical_firms)
    if matches:
        print(f"Title: {title[:50]}...")
        for m in matches:
            print(f"  -> Matched: {m['firm']} (in {m['location']}, variant: '{m['matched_text']}')")
    else:
        print(f"Title: {title[:50]}... -> No matches")
print("-" * 60)

Test matches:
------------------------------------------------------------
Title: Deep Learning for Drug Discovery at Pfizer...
  -> Matched: Pfizer (in both, variant: 'pfizer')
  -> Matched: Wyeth is now a wholly owned subsidiary of Pfizer (in both, variant: 'pfizer')
  -> Matched: Array Biopharma, now a wholly owned subsidiary of Pfizer (in both, variant: 'pfizer')
  -> Matched: Hospira, now a wholly owned subsidiary of Pfizer (in both, variant: 'pfizer')
  -> Matched: Seagen, a wholly owned subsidiary of Pfizer (in both, variant: 'pfizer')
Title: Neural Networks for Genomics...
  -> Matched: GlaxoSmithKline (in abstract, variant: 'glaxosmithkline')
  -> Matched: Novartis (in abstract, variant: 'novartis')
  -> Matched: Human Genome Sciences Inc., a GSK Company (in abstract, variant: 'glaxosmithkline')
  -> Matched: Sierra Oncology LLC - a GSK company (in abstract, variant: 'glaxosmithkline')
  -> Matched: Bellus Health Inc. - a GSK company (in abstract, variant: 'glaxosmithkline')
T

## 7. Run Matching on All NeurIPS Papers

In [ ]:
%%time
# Run matching on all papers
print("Running firm name matching on NeurIPS papers...")

all_matches = []
papers_with_matches = 0

for idx, row in df_neurips.iterrows():
    matches = find_firm_matches(
        row['paper_title'], 
        row['abstract'], 
        canonical_firms
    )
    
    if matches:
        papers_with_matches += 1
        for match in matches:
            all_matches.append({
                'paper_id': row['paper_id'],
                'conference_name': row['conference_name'],
                'year': row['year'],
                'paper_title': row['paper_title'],
                'abstract': row['abstract'],
                'authors': row['authors'],
                'author_order': row['author_order'],
                'author_affiliations': row['author_affiliations'],  # NULL
                'paper_url': row['paper_url'],
                'keywords': row['keywords'],  # NULL
                'acknowledgements': row['acknowledgements'],  # NULL
                'full_text': row['full_text'],  # NULL
                'matched_firm': match['firm'],
                'match_location': match['location'],
                'matched_text': match['matched_text']
            })

print(f"\nMatching complete!")
print(f"Papers with at least one firm match: {papers_with_matches}")
print(f"Total firm-paper matches: {len(all_matches)}")

## 8. Filter to Matched Papers Only

In [9]:
# Create DataFrame of matched papers
df_matches = pd.DataFrame(all_matches)

if len(df_matches) > 0:
    print(f"Matched papers DataFrame shape: {df_matches.shape}")
    print("\n" + "=" * 60)
    print("MATCHING RESULTS SUMMARY")
    print("=" * 60)
    
    # Unique papers
    unique_papers = df_matches['paper_id'].nunique()
    print(f"\nUnique papers with firm matches: {unique_papers}")
    print(f"Out of {len(df_neurips)} total papers ({100*unique_papers/len(df_neurips):.2f}%)")
    
    # Unique firms
    unique_firms = df_matches['matched_firm'].nunique()
    print(f"\nUnique firms matched: {unique_firms}")
    
    # Match location distribution
    print("\nMatch locations:")
    print(df_matches['match_location'].value_counts())
    
    # Matches by year
    print("\nMatches by year:")
    year_summary = df_matches.groupby('year').agg({
        'paper_id': 'nunique',
        'matched_firm': 'count'
    }).rename(columns={'paper_id': 'unique_papers', 'matched_firm': 'total_matches'})
    print(year_summary)
    
    # Top matched firms
    print("\nTop 20 matched firms:")
    print(df_matches['matched_firm'].value_counts().head(20))
    
    # Sample matches
    print("\n" + "-" * 60)
    print("Sample matches:")
    print("-" * 60)
    display(df_matches[['paper_title', 'matched_firm', 'match_location', 'matched_text']].head(10))
else:
    print("No firm matches found in NeurIPS papers.")
    print("This may indicate that clinical trial sponsors are not directly mentioned in conference paper titles/abstracts.")

No firm matches found in NeurIPS papers.
This may indicate that clinical trial sponsors are not directly mentioned in conference paper titles/abstracts.


## 9. Create Database Tables

Connect to MISR database and create the required tables.

In [10]:
# Initialize database connection
# NOTE: Ensure you are connected to UBC VPN before running this cell!

try:
    db = MySQLDatabase()
    
    # Test connection
    with db.get_connection() as conn:
        if conn.is_connected():
            print("Successfully connected to MISR database")
            print(f"Database: {db.config['database']}")
            print(f"Host: {db.config['host']}")
            
            # List existing tables
            tables = db.get_tables()
            print(f"\nExisting tables: {len(tables)}")
except Exception as e:
    print(f"Connection failed: {e}")
    print("\nPlease ensure you are connected to UBC VPN!")

Error connecting to MySQL: 2003 (HY000): Can't connect to MySQL server on 'misr.sauder.ubc.ca:3306' (60)
Connection failed: 2003 (HY000): Can't connect to MySQL server on 'misr.sauder.ubc.ca:3306' (60)

Please ensure you are connected to UBC VPN!


In [ ]:
# SQL to create NeurIPS_Clinical_Trials table (matched papers only)
# NOTE: We are NOT uploading all papers, only those with firm matches

create_matched_papers_sql = """
CREATE TABLE IF NOT EXISTS NeurIPS_Clinical_Trials (
    match_id INT AUTO_INCREMENT PRIMARY KEY,
    paper_id INT,
    conference_name VARCHAR(50),
    year INT,
    paper_title TEXT,
    abstract TEXT,
    authors JSON,
    author_order JSON,
    author_affiliations JSON,
    paper_url VARCHAR(500),
    keywords TEXT,
    acknowledgements TEXT,
    full_text LONGTEXT,
    matched_firm VARCHAR(255),
    match_location VARCHAR(50),
    matched_text TEXT,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    INDEX idx_paper_id (paper_id),
    INDEX idx_firm (matched_firm),
    INDEX idx_year (year),
    INDEX idx_conference_year (conference_name, year)
)
"""

# Create table
try:
    db.execute_update(create_matched_papers_sql)
    print("Created/verified NeurIPS_Clinical_Trials table")
except Exception as e:
    print(f"Error creating table: {e}")

## 10. Upload Matched Papers to Database

Only papers with firm matches are uploaded to `NeurIPS_Clinical_Trials`.

In [ ]:
# Upload matched papers only
if len(df_matches) > 0:
    df_matches_upload = df_matches.copy()
    
    print(f"Uploading {len(df_matches_upload)} matched records to NeurIPS_Clinical_Trials...")
    
    try:
        # Clear existing data first (to allow re-runs)
        db.execute_update("TRUNCATE TABLE NeurIPS_Clinical_Trials")
        print("Cleared existing NeurIPS_Clinical_Trials data")
        
        # Upload in chunks
        rows = db.bulk_insert_dataframe(df_matches_upload, 'NeurIPS_Clinical_Trials', 
                                        if_exists='append', chunksize=500)
        print(f"Successfully uploaded {len(df_matches_upload)} rows to NeurIPS_Clinical_Trials")
    except Exception as e:
        print(f"Error uploading: {e}")
else:
    print("No firm matches found - nothing to upload.")
    print("This may indicate that clinical trial sponsors are not directly mentioned in NeurIPS paper titles/abstracts.")

## 11. Verification

In [14]:
# Verify data was uploaded correctly
print("=" * 60)
print("DATABASE VERIFICATION")
print("=" * 60)

try:
    # Check NeurIPS_Papers
    count_papers = db.execute_query("SELECT COUNT(*) as count FROM NeurIPS_Papers")
    print(f"\nNeurIPS_Papers table: {count_papers['count'][0]:,} rows")
    
    # Year distribution
    year_dist = db.execute_query("""
        SELECT year, COUNT(*) as count 
        FROM NeurIPS_Papers 
        GROUP BY year 
        ORDER BY year
    """)
    print("\nPapers by year in database:")
    display(year_dist)
    
    # Check NeurIPS_Clinical_Trials
    count_matches = db.execute_query("SELECT COUNT(*) as count FROM NeurIPS_Clinical_Trials")
    print(f"\nNeurIPS_Clinical_Trials table: {count_matches['count'][0]:,} rows")
    
    if count_matches['count'][0] > 0:
        # Top matched firms
        top_firms = db.execute_query("""
            SELECT matched_firm, COUNT(*) as match_count
            FROM NeurIPS_Clinical_Trials
            GROUP BY matched_firm
            ORDER BY match_count DESC
            LIMIT 20
        """)
        print("\nTop matched firms in database:")
        display(top_firms)
        
        # Sample records
        sample = db.execute_query("""
            SELECT paper_title, matched_firm, match_location, year
            FROM NeurIPS_Clinical_Trials
            LIMIT 10
        """)
        print("\nSample matched records:")
        display(sample)
        
except Exception as e:
    print(f"Error during verification: {e}")

print("\n" + "=" * 60)
print("Pipeline complete!")
print("=" * 60)

DATABASE VERIFICATION


KeyboardInterrupt: 

## 12. Summary

### Table Created in MISR Database:

**NeurIPS_Clinical_Trials**: Papers where clinical trial sponsor firms were mentioned (MATCHED PAPERS ONLY)

| Column | Description |
|--------|-------------|
| match_id | Auto-increment primary key |
| paper_id | Paper identifier (from source) |
| conference_name | "NeurIPS" |
| year | Publication year (2008-2022) |
| paper_title | Full paper title |
| abstract | Paper abstract (NULL if unavailable) |
| authors | JSON array of author names |
| author_order | JSON array of author positions |
| author_affiliations | NULL (not available in source) |
| paper_url | Link to NeurIPS paper page |
| keywords | NULL (not available in source) |
| acknowledgements | NULL (not available in source) |
| full_text | NULL (not available in source) |
| matched_firm | Clinical trial sponsor name that was matched |
| match_location | Where match was found: "title", "abstract", or "both" |
| matched_text | The specific text variant that matched |

### Data Quality (Prior to Analysis):
- **Total NeurIPS papers (2008-2022)**: 13,405
- **Available fields**: year (100%), paper_title (100%), authors (100%), abstract (99.4%)
- **NULL fields**: author_affiliations, keywords, acknowledgements, full_text

### Matching Methodology:
- Direct firm name matching in paper titles and abstracts
- Known pharma dictionary with verified aliases
- Generic word filtering to reduce false positives
- Minimum match length: 6 characters with word boundary matching